# LAB 4: Implementing a Vietnamese Spell-checking model using PhoBERT

## 1. Introduction


This lab demonstrates how to leverage PhoBERT (Vietnamese BERT) for intelligent spell-checking by using masked language modeling. Instead of relying on simple dictionary lookups, we use the contextual understanding capabilities of BERT to identify potential spelling errors by measuring how well each word fits within its surrounding context.

In [1]:
# Install libraries here (only needed the first time)
%pip install -q transformers torch


Note: you may need to restart the kernel to use updated packages.


## 2. Idea

### 2.1. Spell-checking

The idea to spell-check using BERT is that you will mask each word, one by one, in the sentences and let BERT predict the probability of the word X being the masked word. From that probability, if it is lower than a predetermined threshold, it will be marked as a spelling error.

**Example sentence:**
> "Bánh mì là một loại đồ án của Việt Nam với lớp vỏ ngoài giòn tan, ruột mềm, còn bên trong là phần nhân."

#### Step 1: Masking
Now we will mask the first word, the sentence becomes:

> "[MASK] mì là một loại đồ án của Việt Nam với lớp vỏ ngoài giòn tan, ruột mềm, còn bên trong là phần nhân."

#### Step 2: Calculate probability and determine
Now we will check if **'Bánh'**, the original first word, fits in the masked token.

**Assumption:** The probability for that is `0.927`

Now we can safely say that **'Bánh'** is not a spelling or grammar mistake. 
*(Note: The threshold value of 0.927 being acceptable, you will have to figure out the sweetpoint on your own when you the lab)*

#### Step 3: Repeat the process
You can apply the same technique for the 2nd, 3rd, 4th word... up to the 7th word. 

**Expected result:** The probability for **'án'** should be very low since it should be **'ăn'**. Based on this low probability, you can determine it is a mistake.

### 2.2. Expansion: Find correct word if spelling error

Using libraries such as **difflib** (Python built-in) or **Levenshtein distance** algorithms, you could suggest a word that fits better if it's actually a mistake.

**Implementation approach for the above example:**

1. **Calculate similarity scores:** The model combines and compares words that are most similar to the original word
2. **Test probability fit:** Check if similar words have higher probability to fit the mask position  
3. **Fallback to BERT suggestions:** If similarity-based corrections also fail, use BERT's most suggested word for the masked position

The idea to implement this expansion above can be seen as baseline, as there are many rooms for you to implement a better algorithm. Coding is about creative, there is no such thing as "absolute true answer", especially in AI. So be humble, be hungry, and be creative.

## 3. Helpful tools

To aid your LAB, we have prepared some coding samples so that you can adapt to build a spell-checking model

### 3.1. A masking loop

In [2]:
import re
from typing import List, Tuple

def mask_words_sequentially(sentence: str) -> List[Tuple[str, str, int]]:
    """
    Masks each word in a sentence sequentially for spell-checking.
    
    Args:
        sentence (str): Input sentence to process
        
    Returns:
        List[Tuple[str, str, int]]: List of (masked_sentence, original_word, word_position)
    """
    # Tokenize the sentence (simple word-based tokenization)
    words = sentence.split()
    results = []
    
    for i, word in enumerate(words):
        # Create a copy of words list
        masked_words = words.copy()
        # Replace the current word with [MASK]
        masked_words[i] = "[MASK]"
        # Join back to sentence
        masked_sentence = " ".join(masked_words)
        
        results.append((masked_sentence, word, i))
        print(f"Position {i+1}: {masked_sentence}")
        print(f"Original word: '{word}'")
        print("-" * 50)
    
    return results

# Test the function
test_sentence = "Bánh mì là một loại đồ án của Việt Nam với lớp vỏ ngoài giòn tan"
masked_results = mask_words_sequentially(test_sentence)

Position 1: [MASK] mì là một loại đồ án của Việt Nam với lớp vỏ ngoài giòn tan
Original word: 'Bánh'
--------------------------------------------------
Position 2: Bánh [MASK] là một loại đồ án của Việt Nam với lớp vỏ ngoài giòn tan
Original word: 'mì'
--------------------------------------------------
Position 3: Bánh mì [MASK] một loại đồ án của Việt Nam với lớp vỏ ngoài giòn tan
Original word: 'là'
--------------------------------------------------
Position 4: Bánh mì là [MASK] loại đồ án của Việt Nam với lớp vỏ ngoài giòn tan
Original word: 'một'
--------------------------------------------------
Position 5: Bánh mì là một [MASK] đồ án của Việt Nam với lớp vỏ ngoài giòn tan
Original word: 'loại'
--------------------------------------------------
Position 6: Bánh mì là một loại [MASK] án của Việt Nam với lớp vỏ ngoài giòn tan
Original word: 'đồ'
--------------------------------------------------
Position 7: Bánh mì là một loại đồ [MASK] của Việt Nam với lớp vỏ ngoài giòn tan
Origina

### 3.2. Using PhoBERT to predict the probability of a word

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM
import numpy as np
import os

# Clear any existing HF token to avoid 401 errors
os.environ.pop('HUGGINGFACE_HUB_TOKEN', None)
os.environ.pop('HF_TOKEN', None)

# Try loading models without authentication - prioritizing Vietnamese models
print("Loading BERT model and tokenizer (without authentication)...")

try:
    # PHOBERT_MODEL can point to a local copy of vinai/phobert-base (e.g. ../models/phobert-base)
    PHOBERT_MODEL = os.environ.get("PHOBERT_MODEL", "vinai/phobert-base")
    print(f"Trying PhoBERT base (Vietnamese-specific model): {PHOBERT_MODEL}")
    tokenizer = AutoTokenizer.from_pretrained(PHOBERT_MODEL, token=False)
    model = AutoModelForMaskedLM.from_pretrained(PHOBERT_MODEL, token=False)
    print(f"✅ PhoBERT base loaded successfully!")
except Exception as e:
    print(f"❌ Failed to load PhoBERT base: {e}")
    try:
        print("Trying PhoBERT large...")
        tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-large", token=False)
        model = AutoModelForMaskedLM.from_pretrained("vinai/phobert-large", token=False)
        print(f"✅ PhoBERT large loaded successfully!")
    except Exception as e2:
        print(f"❌ Failed to load PhoBERT large: {e2}")
        try:
            print("Trying alternative Vietnamese BERT model...")
            tokenizer = AutoTokenizer.from_pretrained("NlpHUST/vibert-base-cased", token=False)
            model = AutoModelForMaskedLM.from_pretrained("NlpHUST/vibert-base-cased", token=False)
            print(f"✅ ViBERT model loaded successfully!")
        except Exception as e3:
            print(f"❌ Failed to load ViBERT: {e3}")
            try:
                print("Trying multilingual BERT as fallback...")
                tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased", token=False)
                model = AutoModelForMaskedLM.from_pretrained("bert-base-multilingual-cased", token=False)
                print(f"✅ Multilingual BERT loaded successfully!")
            except Exception as e4:
                print(f"❌ Failed to load multilingual BERT: {e4}")
                try:
                    print("Trying DistilBERT multilingual as final fallback...")
                    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-multilingual-cased", token=False)
                    model = AutoModelForMaskedLM.from_pretrained("distilbert-base-multilingual-cased", token=False)
                    print(f"✅ DistilBERT multilingual loaded successfully!")
                except Exception as e5:
                    print(f"❌ All model loading attempts failed: {e5}")
                    raise Exception("Could not load any BERT model. Please check your internet connection.")

print(f"Model type: {type(model)}")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Tokenizer vocabulary size: {len(tokenizer)}")
print(f"Mask token: {tokenizer.mask_token}")
print(f"Mask token ID: {tokenizer.mask_token_id}")

# Test tokenization with Vietnamese text
test_text = "Bánh mì là món ăn Việt Nam"
tokens = tokenizer.tokenize(test_text)
print(f"Tokenization test: '{test_text}' -> {tokens}")

# Check if this is a Vietnamese-specific model
model_name = model.config.name_or_path if hasattr(model.config, 'name_or_path') else str(type(model))
if 'phobert' in model_name.lower() or 'vibert' in model_name.lower():
    print(f"🇻🇳 Using Vietnamese-specific model: {model_name}")
    print("📈 This should give better results for Vietnamese text!")
else:
    print(f"🌍 Using multilingual model: {model_name}")
    print("📝 This will work with Vietnamese but may not be as accurate as Vietnamese-specific models.")

Loading BERT model and tokenizer (without authentication)...
Trying PhoBERT base (Vietnamese-specific model): /home/user/LAB/models/phobert-base


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie lm_head.bias to lm_head.decoder.bias, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


[transformers] RobertaForMaskedLM LOAD REPORT from: /home/user/LAB/models/phobert-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ PhoBERT base loaded successfully!
Model type: <class 'transformers.models.roberta.modeling_roberta.RobertaForMaskedLM'>
Number of parameters: 135,127,810
Tokenizer vocabulary size: 64001
Mask token: <mask>
Mask token ID: 64000
Tokenization test: 'Bánh mì là món ăn Việt Nam' -> ['Bánh', 'mì', 'là', 'món', 'ăn', 'Việt', 'Nam']
🇻🇳 Using Vietnamese-specific model: /home/user/LAB/models/phobert-base
📈 This should give better results for Vietnamese text!


In [4]:
# Define the functions for spell checking using the loaded BERT model
from typing import List, Tuple

def calculate_word_probability(masked_sentence: str, original_word: str) -> float:
    """
    Calculate the probability of a word fitting in a masked position using BERT.
    
    Args:
        masked_sentence (str): Sentence with [MASK] token
        original_word (str): The original word to check probability for
        
    Returns:
        float: Probability score between 0 and 1
    """
    try:
        # Replace [MASK] with the actual mask token from the tokenizer
        masked_sentence = masked_sentence.replace("[MASK]", tokenizer.mask_token)
        
        # Tokenize the masked sentence
        inputs = tokenizer(masked_sentence, return_tensors="pt", padding=True, truncation=True)
        
        # Find the position of the mask token
        mask_token_id = tokenizer.mask_token_id
        mask_positions = (inputs['input_ids'] == mask_token_id).nonzero(as_tuple=True)[1]
        
        if len(mask_positions) == 0:
            print(f"⚠️ No mask token found in: {masked_sentence}")
            return 0.0
            
        mask_position = mask_positions[0]
        
        # Get model predictions
        with torch.no_grad():
            outputs = model(**inputs)
            predictions = outputs.logits
        
        # Get probabilities for the masked position
        mask_predictions = predictions[0, mask_position]
        probabilities = torch.softmax(mask_predictions, dim=-1)
        
        # Tokenize the original word to get its token ID(s)
        original_word_tokens = tokenizer.encode(original_word, add_special_tokens=False)
        
        # Handle subword tokenization - calculate average probability for all subword tokens
        if original_word_tokens:
            total_prob = 0.0
            for token_id in original_word_tokens:
                total_prob += probabilities[token_id].item()
            probability = total_prob / len(original_word_tokens)
        else:
            print(f"⚠️ Could not tokenize word: {original_word}")
            probability = 0.0
        
        return probability
    
    except Exception as e:
        print(f"❌ Error calculating probability for '{original_word}': {e}")
        return 0.0

sentences = "Bánh mỳ là một loại đồ [MASK] ngon."
prob_ok = calculate_word_probability(sentences, 'ăn')
prob_wrong = calculate_word_probability(sentences, 'án')

print(f"Probability for 'ăn' ", prob_ok)
print(f"Probability for 'án' ", prob_wrong)


Probability for 'ăn'  0.0005241222679615021
Probability for 'án'  1.1411807321337619e-07


### 3.3. Using difflib for similarity checking

In [5]:
import difflib
from typing import List
import unicodedata

def remove_accents(text: str) -> str:
    """Remove Vietnamese accents for similarity comparison."""
    return ''.join(c for c in unicodedata.normalize('NFD', text) 
                   if unicodedata.category(c) != 'Mn')

def find_similar_words(target_word: str, word_list: List[str], n_similar: int = 5) -> List[Tuple[str, float]]:
    """
    Find similar words using difflib sequence matching and phonetic similarity.
    
    Args:
        target_word (str): The word to find similarities for
        word_list (List[str]): List of candidate words
        n_similar (int): Number of similar words to return
        
    Returns:
        List[Tuple[str, float]]: List of (similar_word, similarity_score)
    """
    similarities = []
    target_no_accent = remove_accents(target_word.lower())
    
    for word in word_list:
        if word == target_word:
            continue
            
        word_no_accent = remove_accents(word.lower())
        
        # Calculate similarity using different methods
        # 1. Direct string similarity
        direct_sim = difflib.SequenceMatcher(None, target_word.lower(), word.lower()).ratio()
        
        # 2. Accent-insensitive similarity
        accent_sim = difflib.SequenceMatcher(None, target_no_accent, word_no_accent).ratio()
        
        # 3. Character-level similarity (for typos)
        char_sim = difflib.SequenceMatcher(None, target_word, word).ratio()
        
        # Combine similarities with weights
        combined_similarity = (direct_sim * 0.4 + accent_sim * 0.4 + char_sim * 0.2)
        
        similarities.append((word, combined_similarity))
    
    # Sort by similarity score and return top n_similar
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:n_similar]

def suggest_corrections(misspelled_word: str, masked_sentence: str, 
                       vocabulary: List[str], n_suggestions: int = 3) -> List[Tuple[str, float, float]]:
    """
    Suggest corrections for a misspelled word by combining similarity and probability.
    
    Args:
        misspelled_word (str): The potentially misspelled word
        masked_sentence (str): Sentence with [MASK] where the word should be
        vocabulary (List[str]): List of correct Vietnamese words
        n_suggestions (int): Number of suggestions to return
        
    Returns:
        List[Tuple[str, float, float]]: List of (suggested_word, similarity_score, bert_probability)
    """
    # Find similar words
    similar_words = find_similar_words(misspelled_word, vocabulary, n_similar=10)
    
    suggestions = []
    for word, similarity in similar_words:
        # Calculate BERT probability for each similar word
        prob = calculate_word_probability(masked_sentence, word)
        suggestions.append((word, similarity, prob))
    
    # Sort by combined score (similarity + probability)
    suggestions.sort(key=lambda x: (x[1] * 0.3 + x[2] * 0.7), reverse=True)
    
    return suggestions[:n_suggestions]

# Example Vietnamese vocabulary (in practice, you'd load this from a comprehensive dictionary)
vietnamese_vocab = [
    "bánh", "mì", "là", "một", "loại", "đồ", "ăn", "của", "việt", "nam",
    "với", "lớp", "vỏ", "ngoài", "giòn", "tan", "ruột", "mềm", "còn", 
    "bên", "trong", "phần", "nhân", "thịt", "pate", "rau", "cần", "tây"
]

# Test the similarity functions
print("Testing similarity detection:")
test_word = "án"  # This should be "ăn"
similar = find_similar_words(test_word, vietnamese_vocab, n_similar=5)

print(f"Similar words to '{test_word}':")
for word, similarity in similar:
    print(f"  {word}: {similarity:.4f}")

print("\nTesting correction suggestions:")
masked_sent = "Bánh mì là một loại đồ [MASK] của Việt Nam"
corrections = suggest_corrections("án", masked_sent, vietnamese_vocab, n_suggestions=3)

print(f"Correction suggestions for 'án':")
for word, sim, prob in corrections:
    print(f"  {word}: similarity={sim:.4f}, probability={prob:.4f}, combined={sim*0.3 + prob*0.7:.4f}")

Testing similarity detection:
Similar words to 'án':
  ăn: 0.7000
  bánh: 0.6667
  tan: 0.5600
  cần: 0.5600
  phần: 0.4667

Testing correction suggestions:


Correction suggestions for 'án':
  ăn: similarity=0.7000, probability=0.4327, combined=0.5129
  bánh: similarity=0.6667, probability=0.0255, combined=0.2178
  cần: similarity=0.5600, probability=0.0000, combined=0.1680


## 4. References

[1] Nguyen, D. Q., Nguyen, A. T., Nguyen, D. Q., & Pham, S. B. (2020). PhoBERT: Pre-trained language models for Vietnamese. Findings of the Association for Computational Linguistics: EMNLP 2020, 1037–1042. https://doi.org/10.18653/v1/2020.findings-emnlp.90
[2] Nguyen, V. T., Nguyen, D. Q., Nguyen, K. D., & Nguyen, H. T. (2021). VSEC: A dataset for Vietnamese spelling error correction. Proceedings of the 8th International Workshop on Vietnamese Language and Speech Processing (VLSP 2021). Retrieved from https://github.com/VSEC2021/VSEC